# 🧠 NeuroDSL – Stirling Numbers & Bell Numbers

This notebook demonstrates **NeuroDSL**, a Julia DSL for declarative,
recursive computation graphs with **automatic memoisation**.

We compute **Stirling numbers of the second kind** and the **Bell number** $ B_6 $,
then visualise the entire computation graph interactively.

---

## 📐 Recurrence

Stirling numbers of the second kind \( S(n,k) \) satisfy:

$$
S(n,k) =
\begin{cases}
1 & \text{if } n = 0,\; k = 0,\\[4pt]
0 & \text{if } k = 0 \text{ or } k > n,\\[4pt]
k \cdot S(n-1,k) + S(n-1,k-1) & \text{otherwise}.
\end{cases}
$$

The **Bell number**  $\displaystyle B_n = \sum_{k=1}^{n} S(n,k) $ counts all partitions of an $ n$-element set.

For $ n = 6 $, $ B_6 = 203 $.

---


# Installation (run once)

In [8]:
using NeuroDSL, Printf

# Define custom operators with @defop

In [9]:

@defop scale_add out = attrs[:factor] * inputs[1] + inputs[2]
@defop identity  out = inputs[1]
@defop nsum (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    copyto!(out, inputs[1])
    for i in 2:length(inputs)
        out .+= inputs[i]
    end
end

✅ Op :scale_add registered
✅ Op :identity registered
✅ Op :nsum registered


# Build the recursive Stirling graph

In [10]:
function build_stirling_graph(g, n)
    builder = @neuro g ns=:stirling operators=[:scale_add] begin
        @node one  = 1.0
        @node zero = 0.0

        @rule stirling(n::Int, k::Int) = begin
            if n == 0 && k == 0
                :one
            elseif k == 0 || k > n
                :zero
            else
                scale_add(factor=k, stirling(n-1, k), stirling(n-1, k-1))
            end
        end

        # Compute all S(n,k) for k=1..n
        all_terms = [stirling(n, i) for i in 1:n]
        @node bell = nsum(all_terms...)
    end
    return builder
end

n = 6
g = NeuroGraph(namespace=:stirling, device=Backend.CPUDevice())
builder = build_stirling_graph(g, n)

GraphBuilder(NeuroGraph(Dict{Symbol, Dict{Symbol, GraphNode{Float32}}}(:stirling => Dict(:stirling_27 => GraphNode{Float32}(:stirling_27, nothing, nothing, false, false, Quantom, false, :stirling, Dict{Symbol, Any}(:label => "stirling(5, 2)", :is_rule => true), Symbol[], nothing), :stirling_25 => GraphNode{Float32}(:stirling_25, nothing, nothing, false, false, Quantom, false, :stirling, Dict{Symbol, Any}(:label => "stirling(4, 2)", :is_rule => true), Symbol[], nothing), :scale_add_14 => GraphNode{Float32}(:scale_add_14, nothing, nothing, false, false, Quantom, false, :stirling, Dict{Symbol, Any}(:label => "scale_add_14 = 1·stirling(4, 1) + zero"), Symbol[], nothing), :scale_add_8 => GraphNode{Float32}(:scale_add_8, nothing, nothing, false, false, Quantom, false, :stirling, Dict{Symbol, Any}(:label => "scale_add_8 = 1·stirling(2, 1) + zero"), Symbol[], nothing), :scale_add_40 => GraphNode{Float32}(:scale_add_40, nothing, nothing, false, false, Quantom, false, :stirling, Dict{Symbol, Any

# Execute forward pass

In [11]:
log = ExecutionLog()
ctx = CtxStore()
NeuroDSL.demand!(g, :bell; ctx_store=ctx, namespace=:stirling, log=log)

bell_val = Array(NeuroDSL.node(g, :bell; namespace=:stirling).value)[]
@printf "Bell number B_%d = %d\n" n Int(bell_val)

Bell number B_6 = 203


 # Generate interactive HTML trace

In [12]:
save_interactive_graph(g, log, "stirling_bell_6.html";
                       title = "Stirling & Bell numbers (n=$n)")

✅ Interactive Trace exported → stirling_bell_6.html


In [13]:
# Windows
run(`cmd /c start stirling_bell_6.html`)



Process(`cmd /c start stirling_bell_6.html`, ProcessExited(0))